# tiny-infer: kernel rerun (~30 min on a T4)
Re-runs only the steps that need the Project 1 kernels, plus the correctness oracle with saved output.
Use a **fresh** runtime (Runtime → Disconnect and delete runtime), so vLLM's PyTorch from the first run is gone.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os
if not os.path.exists("tiny-infer"):
    !git clone -q https://github.com/Musharaf-khan/tiny-infer.git
%cd tiny-infer
!git pull -q
!pip install -q -r requirements.txt

In [ ]:
# 1. Build the kernels and check every one against the PyTorch reference (must print PASS)
!python -m tiny_infer.kernels

In [ ]:
# 2. Correctness oracle on TinyLlama (results/correctness_*.json)
!python scripts/check_correctness.py --dtype float32
!python scripts/check_correctness.py --dtype float16
!python scripts/check_correctness.py --dtype float16 --custom-kernels

In [ ]:
# 3. Serving sweep with the kernels (same workload as the first run's 'engine' rows)
!python scripts/run_serving_bench.py --label engine+kernels --server "python -m tiny_infer.server --max-num-seqs 128 --custom-kernels" -- --concurrency 1 4 16 32 64

In [ ]:
# 4. Op-level and end-to-end kernel deltas
!python bench/kernel_bench.py --e2e-batches 1 16 32 64

In [ ]:
# 5. Where a decode step goes, without and with kernels
!python bench/step_profile.py --batch 32
!python bench/step_profile.py --batch 32 --custom-kernels
!python bench/step_profile.py --batch 64 --custom-kernels

In [ ]:
!zip -qr results_kernels.zip results
from google.colab import files; files.download("results_kernels.zip")